In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

#!pip install xgboost pandas sklearn

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 

In [ ]:
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
#display(Water_Quality_df.head(5))

In [ ]:
landsat_train_features = pd.read_csv("landsat_features_training_v2.csv")
#display(landsat_train_features.head(5))

In [ ]:
landsat_train_features.info()

In [ ]:
landsat_basins_train_df = pd.read_csv("train_with_landsat.csv")
display(landsat_basins_train_df.info())

In [ ]:
# If NDMI and MNDWI columns are of type object, convert them to float
#landsat_train_features['NDMI'] = landsat_train_features['NDMI'].astype(float)
#landsat_train_features['MNDWI'] = landsat_train_features['MNDWI'].astype(float)

In [ ]:
Terraclimate_df = pd.read_csv("terraclimate_features_training_v3.csv")
display(Terraclimate_df.head(5))

In [ ]:
hydrobasins_train_df = pd.read_csv("hydrobasins_features_train.csv")
display(hydrobasins_train_df.head(5))


In [5]:
# Combine two datasets vertically (along columns) using pandas concat function.
def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [6]:
# Combining ground data and final data into a single dataset.
wq_data_aux = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
#display(wq_data.head(5))
wq_data = pd.concat([wq_data_aux,hydrobasins_train_df], axis=1)
wq_data = wq_data.loc[:, ~wq_data.columns.duplicated()]

In [7]:
display(wq_data.isna().sum())
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))

In [9]:
def split_data(X, y, test_size=0.1, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler
    
'''
def train_model(X_train_scaled, y_train):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    return model

'''   
from lightgbm import LGBMRegressor

def train_model(X_train_scaled, y_train):
    lgb_model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    lgb_model.fit(X_train_scaled, y_train)
    return lgb_model

'''
import xgboost as xgb

def train_model(X_train_scaled, y_train):
    dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
    params ={
    "objective":"reg:squarederror", 
    "max_depth":4,
    "eta":0.3,
    "eval_metric":"mlogloss",
    }
    xgb_model = xgb.train(params, dtrain, num_boost_round=50)
    return xgb_model
'''


def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
#comentar la de DMatrix si no es xgboost!!!
    #X_scaled = xgb.DMatrix(X_scaled, label=y_true)
    y_pred = model.predict(X_scaled)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n{dataset_name} Evaluation:")
    print(f"R²: {r2:.3f}")
    print(f"RMSE: {rmse:.3f}")
    return y_pred, r2, rmse

In [10]:
def run_pipeline(X, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    print(f"{'='*60}")
    
    # Split data
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # Scale
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    
    # Train
    model = train_model(X_train_scaled, y_train)
    
    # Evaluate (in-sample)
    y_train_pred, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")
    
    # Evaluate (out-sample)
    y_test_pred, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")
    
    # Return summary
    results = {
        "Parameter": param_name,
        "R2_Train": r2_train,
        "RMSE_Train": rmse_train,
        "R2_Test": r2_test,
        "RMSE_Test": rmse_test
    }
    return model, scaler, pd.DataFrame([results])

In [ ]:
X = wq_data.drop(columns=['Latitude', 'Longitude', 'Sample Date','Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

#solo para escoger variables, se puede comentar entero

base_features = [
     'pet',
    'ppt', 'q', 'soil', 'tmax', 'tmin', 
    'aet', 'month',
    'pet_lag1','ppt_lag1','q_lag1',
    'tmax_lag1','tmin_lag1','aet_lag1','pet_lag2',
    'ppt_lag2','q_lag2','tmax_lag2','tmin_lag2',
    'aet_lag2','pet_lag3','ppt_lag3','q_lag3','tmax_lag3',
    'tmin_lag3','aet_lag3','ppt_roll3_sum',
    'q_roll3_sum','ppt_roll6_sum','q_roll6_sum',
    'ppt_roll12_sum','q_roll12_sum','tmin_roll3_mean',
    'tmax_roll3_mean','soil_roll3_mean','tmin_roll6_mean',
    'tmax_roll6_mean','soil_roll6_mean','tmin_roll12_mean',
    'tmax_roll12_mean','soil_roll12_mean',
    'blue','green','red','nir','swir16','swir22','NDMI','MNDWI','NDVI','EVI',
'SAVI','NMDI','FAI','turbidity','NDWI','red_green','swir_nir','swir2_nir',
'NDTI','BSI','AWEI','SI',
'ppt_mean', 'ppt_3m', 'ppt_6m', 'aet_mean', 'aet_3m', 'aet_6m', 'water_balance_3m', 'water_balance_6m'
]

'''
targets = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]
'''
#wq_data = wq_data[['Latitude', 'Longitude', 'Sample Date']+ base_features + targets]


In [ ]:
y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

In [ ]:
features_TA=[]
corrs = []
from scipy.stats import spearmanr

for feature in base_features:
    valid = pd.concat([X[feature], y_TA], axis=1).dropna()
    rho, p = spearmanr(valid[feature], valid['Total Alkalinity'])
    corrs.append((feature, rho, p))
    if abs(rho)>0.15: features_TA.append(feature)

corr_df = pd.DataFrame(corrs, columns=["feature", "spearman_rho", "p_value"])
corr_df.sort_values("spearman_rho", key=abs, ascending=False)

In [ ]:
corrs = []
features_EC=[]

for feature in base_features:
    valid = pd.concat([X[feature], y_EC], axis=1).dropna()
    rho, p = spearmanr(valid[feature], valid['Electrical Conductance'])
    corrs.append((feature, rho, p))
    if abs(rho)>0.15: features_EC.append(feature)

corr_df = pd.DataFrame(corrs, columns=["feature", "spearman_rho", "p_value"])
corr_df.sort_values("spearman_rho", key=abs, ascending=False)

In [ ]:
corrs = []
features_DRP = []

for feature in base_features:
    valid = pd.concat([X[feature], y_DRP], axis=1).dropna()
    rho, p = spearmanr(valid[feature], valid['Dissolved Reactive Phosphorus'])
    corrs.append((feature, rho, p))
    if abs(rho)>0.1: features_DRP.append(feature)

corr_df = pd.DataFrame(corrs, columns=["feature", "spearman_rho", "p_value"])
corr_df.sort_values("spearman_rho", key=abs, ascending=False)




In [11]:
model_TA, scaler_TA, results_TA = run_pipeline(X[features_TA], y_TA, "Total Alkalinity")
model_EC, scaler_EC, results_EC = run_pipeline(X[features_EC], y_EC, "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X[features_DRP], y_DRP, "Dissolved Reactive Phosphorus")

In [12]:
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
results_summary

In [ ]:
test_file = pd.read_csv("submission_template.csv")
#display(test_file.head(5))

In [ ]:
landsat_val_features = pd.read_csv("landsat_features_validation_v2.csv")
#display(landsat_val_features.head(5))

In [ ]:
Terraclimate_val_df = pd.read_csv("terraclimate_features_validation_v3.csv")
#display(Terraclimate_val_df.head(5))

In [ ]:
hydrobasins_val_df = pd.read_csv("hydrobasins_features_validation.csv")

In [16]:
#Consolidate all the extracted bands and features in a single dataframe
val_data = landsat_val_features.merge(
    Terraclimate_val_df,
    on=['Latitude', 'Longitude', 'Sample Date'],
    how='inner'
).merge(
    hydrobasins_val_df,
    on=['Latitude', 'Longitude', 'Sample Date'],
    how='left'
)

submission_val_data=val_data.copy()
submission_val_data=submission_val_data.drop(columns=['Latitude', 'Longitude', 'Sample Date'])

# Impute the missing values, if any
submission_val_data = submission_val_data.fillna(submission_val_data.median(numeric_only=True))

display(submission_val_data.info())


In [19]:
submission_val_data.shape

In [20]:

# --- Predicting for Total Alkalinity ---
X_sub_scaled_TA = scaler_TA.transform(submission_val_data[features_TA])
#Comentar si no es xgboost!!
#X_sub_scaled_TA = xgb.DMatrix(X_sub_scaled_TA)
pred_TA_submission = model_TA.predict(X_sub_scaled_TA)

# --- Predicting for Electrical Conductance ---
X_sub_scaled_EC = scaler_EC.transform(submission_val_data[features_EC])
#Comentar si no es xgboost!!
#X_sub_scaled_EC = xgb.DMatrix(X_sub_scaled_EC)
pred_EC_submission = model_EC.predict(X_sub_scaled_EC)

# --- Predicting for Dissolved Reactive Phosphorus ---
X_sub_scaled_DRP = scaler_DRP.transform(submission_val_data[features_DRP])
#Comentar si no es xgboost!!
#X_sub_scaled_DRP = xgb.DMatrix(X_sub_scaled_DRP)
pred_DRP_submission = model_DRP.predict(X_sub_scaled_DRP)

In [21]:
submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': pred_TA_submission,
    'Electrical Conductance': pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})

In [ ]:
#Dumping the predictions into a csv file.

submission_df.to_csv("submission.csv",index = False)

import base64
from IPython.display import HTML

csv_text = submission_df.to_csv(index=False)
b64 = base64.b64encode(csv_text.encode()).decode()
fname = "submission.csv"

html = f'''
<a download="{fname}" href="data:text/csv;base64,{b64}">
  Descargar {fname}
</a>
'''
HTML(html)